In [7]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader

import datasets

import pie_datasets
from pytorch_ie.models.transformer_span_classification import TransformerSpanClassificationModel
from pytorch_ie.taskmodules.transformer_span_classification import TransformerSpanClassificationTaskModule
from pie_datasets import load_dataset
from pytorch_ie.documents import TextDocumentWithLabeledSpansAndBinaryRelations

import json
import os
from pytorch_ie.annotations import BinaryRelation
import random


In [2]:
NO_RELATION_LABEL = 'no relation'

In [ ]:


def convert_to_training_samples(sample, base_path, data_section, adu_detection_system_message, relation_detection_system_message, relations_labels):
    text = sample.text
    sentences = text.split('.')
    for i, sentence in enumerate(sentences):
        if sentence.strip() == '':
            continue
        start_index = text.index(sentence) - 5 # a little shift to avoid extreme conditions
        end_index = start_index + len(text) + 5
        prompt_messages = [
                {
                    'role': 'system',
                    'content': adu_detection_system_message
                },
                {
                    'role': 'user',
                    'content': text + f'\nWhat argument components exists in sentence \"{sentence}\" from the above dispute? Please annotate the constituing words.'
                },
        ]
        found = False
        assistant_message = sentence[:]
        for span in sample.labeled_spans:
            span_text = str(span).rstrip('.')
            if span_text in sentence and span.start >= start_index and span.end <= end_index:
                found = True
                assistant_message = assistant_message.replace(span_text.strip(), f'<{span.label}>{span_text.strip()}</{span.label}>')
        if not found:
            assistant_message = 'The given sentence is not argumentative.'
        prompt_messages += [
            {
                'role': 'assistant',
                'content': assistant_message
            }
        ]
        with open(f'{base_path}/adus/{data_section}/{sample.id}-{i:02d}.json', 'w') as f:
            json.dump(prompt_messages, f, indent=4)

    relations = sample.binary_relations
    relations_prompt_options = '\n'.join(f'{i+1}. {rl}' for i, rl in enumerate(relations_labels))
    for i, r in enumerate(relations):
        prompt_messages = [
                {
                    'role': 'system',
                    'content': relation_detection_system_message
                },
                {
                    'role': 'user',
                    'content': text + f'\nWhat is the relation between \"{str(r.head)}\" and \"{str(r.tail)}\" from the above essay? Choose from following options  {relations_prompt_options}'
                },
                {
                    'role': 'assistant',
                    'content': f'{relations_labels.index(r.label)+1}. {r.label}'
                }
        ]
        
        with open(f'{base_path}/relations/{data_section}/{sample.id}-{i:02d}.json', 'w') as f:
            json.dump(prompt_messages, f, indent=4)
    
    random_relations = []
    while len(random_relations) <= 3:
        span1 = random.choice(sample.labeled_spans)
        span2 = random.choice(sample.labeled_spans)
        
        if span1 == span2:
            continue
        
        has_relation = False
        for r in relations:
            if r.head == span1 and r.tail == span2: #only one way relation
                has_relation = True
                continue
        if has_relation:
            continue
        prompt_messages = [
                {
                    'role': 'system',
                    'content': relation_detection_system_message
                },
                {
                    'role': 'user',
                    'content': text + f'\nWhat is the relation between \"{str(span1)}\" and \"{str(span2)}\" from the above essay? Choose from following options {relations_prompt_options}'
                },
                {
                    'role': 'assistant',
                    'content':  f'{relations_labels.index(NO_RELATION_LABEL)+1}. {NO_RELATION_LABEL}'
                }
        ]
        random_relations.append(prompt_messages)
        
    for i, r in enumerate(random_relations):
        with open(f'{base_path}/relations/{data_section}/{sample.id}-{i+len(relations):02d}.json', 'w') as f:
            json.dump(r, f, indent=4)

In [9]:
def convert_to_training_samples_oracle(sample, base_path, data_section, adu_detection_system_message, adus_labels):
    all_labels = ' '.join([f'{i+1}. {l}' for i, l in enumerate(adus_labels)])
    for i, span in enumerate(sample.labeled_spans):
        prompt_messages = [
            {
                'role': 'system',
                'content': adu_detection_system_message
            },
            {
                'role': 'user',
                'content': f'\n{sample.text}\nWhat is the argumentative role of "{str(span).strip()}" from the given essay? choose from {all_labels} \n'
            },
            {
                'role': 'assistant',
                'content': f'{adus_labels.index(span.label)+1}. {span.label}',
            }
        ]
        with open(f'{base_path}/adus/{data_section}/{sample.id}-{i:02d}.json', 'w') as f:
            json.dump(prompt_messages, f, indent=4)

# AAE2 (PE2)

In [3]:
adu_detection_system_message = """You will be given an essay and asked about to annotate a specified sentence from it. Annotate the argumentative parts of specified sentence with their argumentative roles such as MajorClaim, Claim and Premise using xml like tags to indicate start and end of a component. Note that most of the sentences are argumentative. Annotate as much as you can."""
relation_detection_system_message = """You will be given an essay and asked about to find the relation (support, attack or no_relation) between two selected argument components from it."""

aae2 = load_dataset(path="pie/aae2", conversion_method="connect_all", revision="pr/3")
aae2 = aae2.to_document_type(TextDocumentWithLabeledSpansAndBinaryRelations)

aae2_train_docs = aae2["train"]
aae2_test_docs = aae2["test"]

Repo card metadata block was not found. Setting CardData to empty.


<class 'datasets_modules.datasets.DFKI-SLT--brat.624e34453f5c35b67763820833773e58012de6bd5bfc3e9216c41550b31a8e2b.brat.BratConfig'>
<class 'datasets_modules.datasets.pie--aae2.bc8667e80b440a7f0776f73d859b6a8476faf0ba8de9979f6571f71948f59e9b.aae2.ArgumentAnnotatedEssaysV2Config'>


In [ ]:
base_path = 'datasets/aae2' 

os.makedirs(f'{base_path}/adus/train', exist_ok=True)
os.makedirs(f'{base_path}/adus/test', exist_ok=True)
os.makedirs(f'{base_path}/relations/train', exist_ok=True)
os.makedirs(f'{base_path}/relations/test', exist_ok=True)

labels_set = set()
for sample in aae2_train_docs:
    labels = [r.label for r in sample.binary_relations]
    labels_set |= set(labels)
labels_set = list(labels_set)
labels_set.append(NO_RELATION_LABEL)
print(labels_set)

for sample in aae2_train_docs:
    convert_to_training_samples(sample, base_path, 'train', adu_detection_system_message, relation_detection_system_message, labels_set)
    
for sample in aae2_test_docs:
    convert_to_training_samples(sample, base_path, 'test', adu_detection_system_message, relation_detection_system_message, labels_set)

['attacks', 'supports', 'no relation']


In [12]:
#Oracle span model
adu_detection_system_message = """You will be given an essay and asked about to annotate a specified part from it. Choose the argumentative role of the given sentence from list 1. MajorClaim, 2. Claim and 3. Premise. Choose only one label."""
base_path = 'datasets/aae2-oracle' 

os.makedirs(f'{base_path}/adus/train', exist_ok=True)
os.makedirs(f'{base_path}/adus/test', exist_ok=True)

adus_labels = []
for sample in aae2_train_docs:
     for i, span in enumerate(sample.labeled_spans):
         adus_labels.append(span.label)
adus_labels = list(set(adus_labels))

for sample in aae2_train_docs:
    convert_to_training_samples_oracle(sample, base_path, 'train', adu_detection_system_message, adus_labels)
    
for sample in aae2_test_docs:
    convert_to_training_samples_oracle(sample, base_path, 'test', adu_detection_system_message, adus_labels)

# ArgMicroText


In [4]:
argmicro_dataset = load_dataset("pie/argmicro", 'en')
argmicro_dataset = argmicro_dataset.to_document_type(TextDocumentWithLabeledSpansAndBinaryRelations)
print(argmicro_dataset)

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['id', 'topic_id', 'stance', 'text', 'edus', 'adus', 'edges', 'metadata', 'relations', 'labeled_spans', 'binary_relations'],
        num_rows: 112
    })
})


In [5]:
argmicro_train_docs = argmicro_dataset["train"][:int(112*.7)]
argmicro_test_docs = argmicro_dataset["train"][int(112*.7):]

In [13]:
import os

base_path = 'datasets/argmicro' 

os.makedirs(f'{base_path}/adus/train', exist_ok=True)
os.makedirs(f'{base_path}/adus/test', exist_ok=True)
os.makedirs(f'{base_path}/relations/train', exist_ok=True)
os.makedirs(f'{base_path}/relations/test', exist_ok=True)

labels_set = set()
for sample in argmicro_train_docs:
    labels = [r.label for r in sample.binary_relations]
    labels_set |= set(labels)
labels_set = list(labels_set)
labels_set.append(NO_RELATION_LABEL)
print(labels_set)

adu_detection_system_message = """
You will be given an argumentation and will be asked about to annotate a specified sentence from it for it's argument components.
Annotate the argumentative parts of specified sentence with their argumentative roles such as pro (proponent), opp (opponent) using xml like tags to indicate start and end of a component.
Note that most of the sentences are argumentative. Annotate as much as you can."""
relation_detection_system_message = """You will be given an argumentation and asked about to find the relation from relation types: sup (support), exa (support by example), add (additional source), reb (rebutting attack), und (undercutting attack), joint (joining relation) between two selected argument components from it."""

for sample in argmicro_train_docs:
    convert_to_training_samples(sample, base_path, 'train', adu_detection_system_message, relation_detection_system_message, labels_set)
    
for sample in argmicro_test_docs:
    convert_to_training_samples(sample, base_path, 'test', adu_detection_system_message, relation_detection_system_message, labels_set)

['reb', 'und', 'joint', 'exa', 'sup', 'no relation']


In [12]:
#Oracle span model
adu_detection_system_message = """You will be given an essay and asked about to annotate a specified part from it. Choose the argumentative role of the given sentence from list 1.opp (opponent), 2.pro (proponent). Choose only one label."""
base_path = 'datasets/argmicro-oracle' 

os.makedirs(f'{base_path}/adus/train', exist_ok=True)
os.makedirs(f'{base_path}/adus/test', exist_ok=True)

adus_labels = []
for sample in argmicro_train_docs:
     for i, span in enumerate(sample.labeled_spans):
         adus_labels.append(span.label)
adus_labels = list(set(adus_labels))

for sample in argmicro_train_docs:
    convert_to_training_samples_oracle(sample, base_path, 'train', adu_detection_system_message, adus_labels)
    
for sample in argmicro_test_docs:
    convert_to_training_samples_oracle(sample, base_path, 'test', adu_detection_system_message, adus_labels)

# Sciarg

In [46]:
from pie_datasets import load_dataset

sciarg_dataset = load_dataset(path="pie/sciarg")
sciarg_dataset = sciarg_dataset.to_document_type(TextDocumentWithLabeledSpansAndBinaryRelations)

sciarg_train_docs = sciarg_dataset["train"][:30]
sciarg_test_docs = sciarg_dataset["train"][30:]

Map: 100%|██████████| 40/40 [00:01<00:00, 29.87 examples/s]


In [49]:
base_path = 'datasets/sciarg' 

os.makedirs(f'{base_path}/train/adus', exist_ok=True)
os.makedirs(f'{base_path}/train/relations', exist_ok=True)
os.makedirs(f'{base_path}/test/adus', exist_ok=True)
os.makedirs(f'{base_path}/test/relations', exist_ok=True)

labels_set = set()
for sample in sciarg_train_docs:
    labels = [r.label for r in sample.binary_relations]
    labels_set |= set(labels)
labels_set = list(labels_set)
labels_set.append(NO_RELATION_LABEL)
print(labels_set)

adu_detection_system_message = """You will be given the abstract and introduction of a paper and will be asked about to annotate a specified sentence from it. Annotate the argumentative parts of specified sentence with their argumentative roles such as MajorClaim, Claim and Premise using xml like tags to indicate start and end of a component. Note that most of the sentences are argumentative. Annotate as much as you can."""
relation_detection_system_message = f"You will be given the abstract and introduction of a paper and asked about to find the relation ({','.join(labels_set)}) between two selected argument components from it."


for sample in sciarg_train_docs:
    convert_to_training_samples(sample, f'{base_path}/train' , adu_detection_system_message, relation_detection_system_message, labels_set )
    
for sample in sciarg_test_docs:
    convert_to_training_samples(sample, f'{base_path}/test', adu_detection_system_message, relation_detection_system_message, labels_set)

['semantically_same', 'contradicts', 'parts_of_same', 'supports', 'no relation']
